# Upload fichier vers Supabase Storage

Ce notebook permet de sélectionner un fichier local et de l'envoyer dans un bucket Supabase Storage.

In [1]:
%pip install "typing-extensions>=4.15.0" supabase python-dotenv -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from supabase import create_client

load_dotenv()

raw_url = os.environ["SUPABASE_URL"]
match = re.search(r"([a-z]{20,})\.supabase\.co", raw_url)
project_ref = match.group(1) if match else None
assert project_ref, f"Impossible d'extraire le project ref depuis SUPABASE_URL ({raw_url})"

api_url = f"https://{project_ref}.supabase.co"

assert "SUPABASE_SERVICE_ROLE_KEY" in os.environ, (
    "SUPABASE_SERVICE_ROLE_KEY manquante dans .env\n"
    "→ Dashboard Supabase > Settings > API > service_role key"
)
api_key = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

supabase = create_client(api_url, api_key)
print(f"Supabase client ready : {api_url}")

Supabase client ready : https://nybxcfjjnkgxzgaeitlk.supabase.co


In [3]:
buckets = supabase.storage.list_buckets()
print("Buckets disponibles :")
for b in buckets:
    print(f"  - {b.name}  (public={b.public})")

Buckets disponibles :
  - farfetch_files  (public=False)


In [ ]:
# ---- Configuration ----
LOCAL_FILE = ""          # ex: "/Users/me/data/export.csv"
BUCKET     = "TEST"          # ex: "farfetch_files"
DEST_PATH  = "imports/test.ipynb"          # ex: "imports/2025/export.csv"

In [5]:
import mimetypes

local = Path(LOCAL_FILE).expanduser().resolve()
assert local.is_file(), f"Fichier introuvable : {local}"
assert BUCKET, "BUCKET ne peut pas être vide"

dest = DEST_PATH or local.name
content_type = mimetypes.guess_type(str(local))[0] or "application/octet-stream"

print(f"Fichier  : {local}  ({local.stat().st_size / 1024:.1f} KB)")
print(f"Bucket   : {BUCKET}")
print(f"Dest     : {dest}")
print(f"Type     : {content_type}")

Fichier  : /Users/alexisflorentin/Documents/Freelancing/Adam Lippes/cron_functions/notebooks/upload_to_supabase_storage.ipynb  (15.3 KB)
Bucket   : TEST
Dest     : imports/test.ipynb
Type     : application/octet-stream


In [6]:
existing = [b.name for b in supabase.storage.list_buckets()]
if BUCKET not in existing:
    supabase.storage.create_bucket(BUCKET, options={"public": False})
    print(f"Bucket '{BUCKET}' créé")

with open(local, "rb") as f:
    res = supabase.storage.from_(BUCKET).upload(
        path=dest,
        file=f,
        file_options={"content-type": content_type},
    )

print("Upload terminé")
print(f"  Bucket : {BUCKET}")
print(f"  Path   : {dest}")
print(f"  Response : {res}")

Bucket 'TEST' créé
Upload terminé
  Bucket : TEST
  Path   : imports/test.ipynb
  Response : UploadResponse(path='imports/test.ipynb', full_path='TEST/imports/test.ipynb', fullPath='TEST/imports/test.ipynb')
